In [1]:
from astroquery.gaia import Gaia

q = """
SELECT TOP 10
    source_id, ra, dec, l, b,
    phot_g_mean_mag, parallax, parallax_error,
    pmra, pmra_error, pmdec, pmdec_error
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 10
"""

job = Gaia.launch_job_async(q)
tab = job.get_results()
tab

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
INFO: Query finished. [astroquery.utils.tap.core]


SOURCE_ID,ra,dec,l,b,phot_g_mean_mag,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error
,deg,deg,deg,deg,mag,mas,mas,mas / yr,mas / yr,mas / yr,mas / yr
int64,float64,float64,float64,float64,float32,float64,float32,float64,float32,float64,float32
92603790642688,44.41500318632742,0.5679005443548504,175.7211091075555,-48.92340236454754,9.920633,10.969874447270998,0.018589059,124.61536790092579,0.018589567,66.85767871994857,0.015076482
349374115419136,44.53332789932286,1.3711480438315549,174.99106558154145,-48.27063379932842,9.519655,6.279127258549036,0.018122906,-1.1783702463560783,0.018652856,-16.487926265046266,0.016640656
673008491088640,46.323755939787574,1.895340177005703,176.30993558430916,-46.632745123566515,8.055367,5.56693233635593,0.17825021,61.673581362405194,0.12743916,-27.210041492302445,0.119145185
695170522295168,46.38030666104432,2.114453435851673,176.14005359094335,-46.43897194752123,9.633048,5.552260708543552,0.0277208,60.77519960463076,0.022481222,-29.254848275595965,0.020826811
804846807203968,46.67693127841008,2.475384325452602,176.06852761252802,-45.974573742856876,9.098787,6.75751170650531,0.0174791,-19.064853082191227,0.019454155,-7.628936411634683,0.018023323
1371026575907456,43.98087656471436,2.631811707884033,173.0901025700839,-47.74457478226914,9.542982,1.7358696325244476,0.07114073,-0.8746130389594635,0.0782143,-9.444539550220272,0.060629573
1546639198757760,42.96991092650169,2.91367584784096,171.6948363170372,-48.220948914367334,8.898553,2.9032836307439642,0.028722532,4.282612491431546,0.032463595,3.0051184601113676,0.026947409
1753553543188992,45.46359917458231,3.138095461386887,174.15201318991387,-46.35683310964348,8.163041,4.017941960688069,0.030786442,1.82787636211724,0.03356609,9.885086354334051,0.028722087


Tegkelidis et al. give the Gaia-registered SN 1987A position as:

RA  = 5h35m27.9884s = 83.866618 deg

Dec = -69°16′11.1134″ = -69.269754 deg

They also use Gaia DR3 as the astrometric reference frame for SN 1987A, so this is an appropriate centre

In [2]:
from astroquery.gaia import Gaia
from pathlib import Path

ra_87a = 83.866618
dec_87a = -69.269754
radius_deg = 0.05  # 3 arcmin test region

q_87a_test = f"""
SELECT
    source_id,
    ra, ra_error,
    dec, dec_error,
    l, b,
    phot_g_mean_mag,
    phot_bp_mean_mag,
    phot_rp_mean_mag,
    bp_rp,
    parallax, parallax_error,
    pmra, pmra_error,
    pmdec, pmdec_error,
    ruwe,
    visibility_periods_used,
    astrometric_params_solved,
    duplicated_source
FROM gaiadr3.gaia_source
WHERE 1 = CONTAINS(
    POINT('ICRS', ra, dec),
    CIRCLE('ICRS', {ra_87a}, {dec_87a}, {radius_deg})
)
AND phot_g_mean_mag < 19
"""

job = Gaia.launch_job_async(q_87a_test)
tab = job.get_results()
df = tab.to_pandas()

print(len(df))
df.head()

INFO: Query finished. [astroquery.utils.tap.core]
917


,SOURCE_ID,ra,ra_error,dec,dec_error,l,b,phot_g_mean_mag,phot_bp_mean_mag,phot_rp_mean_mag,...,parallax,parallax_error,pmra,pmra_error,pmdec,pmdec_error,ruwe,visibility_periods_used,astrometric_params_solved,duplicated_source
0,4657661929683755392,83.834240,0.171331,-69.315687,0.166024,279.758864,-31.942511,18.923672,NaN,NaN,...,-0.174868,0.166782,1.811821,0.258056,0.996704,0.242275,0.983766,27,95,False
1,4657661994074027776,83.876487,0.157982,-69.306279,0.157453,279.745590,-31.928931,18.808245,19.205477,17.702154,...,0.665312,0.146355,-3.467770,0.261153,-11.140910,0.204004,1.034317,26,95,False
2,4657662028455648768,83.823972,0.102288,-69.311471,0.105024,279.754493,-31.946656,18.116598,18.044447,18.088047,...,0.266354,0.103850,1.725402,0.152042,0.550061,0.140174,0.970602,27,95,False
3,4657662032762947072,83.822787,0.058203,-69.307669,0.060453,279.750115,-31.947566,16.769110,16.692026,16.807093,...,0.129299,0.057067,1.987818,0.078067,0.155725,0.079401,1.134648,25,31,False
4,4657662681290482048,83.789429,0.130766,-69.307582,0.124803,279.751825,-31.959264,18.345474,19.022034,17.637592,...,0.314128,0.124417,2.145121,0.176308,0.649705,0.178549,0.983953,23,95,False


Testing if saving it works:

In [4]:
out = Path("input_files") / "SN1987A_gaia_test.csv"
df.to_csv(out, index=False)
print(out.resolve())

C:\Users\bukow\DataspellProjects\87A\input_files\SN1987A_gaia_test.csv


In [5]:
from astroquery.gaia import Gaia

q = """
SELECT TOP 10
    source_id, ra, dec, phot_g_mean_mag, pmra, pmdec, parallax
FROM gaiadr3.gaia_source
WHERE phot_g_mean_mag < 10
"""

job = Gaia.launch_job_async(q)
tab = job.get_results()
tab

INFO: Query finished. [astroquery.utils.tap.core]


SOURCE_ID,ra,dec,phot_g_mean_mag,pmra,pmdec,parallax
,deg,deg,mag,mas / yr,mas / yr,mas
int64,float64,float64,float32,float64,float64,float64
7632157690368,45.034342834696204,0.23538959951604121,8.068802,45.4660098305345,-6.834343096515991,5.60229354383997
564397358129664,46.3291071291336,1.2709899670233795,8.884698,-11.88098012228794,-14.473363354651408,3.4488164175629206
1428785296198272,42.76014595905163,2.1224085974764058,8.340802,4.234726436880372,-9.627608841152332,2.344264668716948
1504716022924544,42.59461202840367,2.443473561751165,8.504659,70.298922550993,-35.803038427641624,7.5317123179852405
1511489186391040,42.59394717767663,2.613397614198138,9.016174,13.431677574448715,8.788631627238573,2.141542234024374
1512966655182592,42.90241554818587,2.4038052035535546,9.049626,-15.957068696598359,-33.82638971331455,7.459379289789435
2050181164601088,44.19458025352181,3.5008800044262776,8.3661785,-3.0656776886179795,8.844728196023175,3.0050335859003794
2761084151394944,47.60656145112092,4.25816028404123,9.082378,14.155892157302763,-4.0298562044052595,7.910022033925035


In [1]:
from input_files import Gaia_DR3_filtering as gdf
from input_files import Gaia_DR3_filtering_nozeropoint as gdf_nozp

print("filtering imports ok")

filtering imports ok
